# 作业1：自动拼写纠错

欢迎来到第2门课程的第一次作业。本次作业能够帮助你巩固 Python 编程与概率相关知识。通过完成本次作业，你将实现一套高效实用的自动拼写纠错系统。

## 大纲
- [0. 概述](#0)
    - [0.1 编辑距离](#0-1)
- [1. 数据预处理](#1)
    - [1.1 练习1](#ex-1)
    - [1.2 练习2](#ex-2)
    - [1.3 练习3](#ex-3)
- [2. 字符串操作](#2)
    - [2.1 练习4](#ex-4)
    - [2.2 练习5](#ex-5)
    - [2.3 练习6](#ex-6)
    - [2.4 练习7](#ex-7)
- [3. 编辑操作组合](#3)
    - [3.1 练习8](#ex-8)
    - [3.2 练习9](#ex-9)
    - [3.3 练习10](#ex-10)
- [4. 最小编辑距离](#4)
    - [4.1 练习11](#ex-11)
- [5. 回溯路径（选做）](#5)

<a name='0'></a>
## 0. 概述

大家每天在手机和电脑上都会使用自动纠错功能。在本次作业中，你将探究它背后真正的实现原理。当然，你即将实现的模型和手机内置的纠错系统并不完全一致，但效果依旧很不错。

完成本次作业后，你将学会：
- 根据语料统计单词出现次数
- 计算语料中单词的出现概率
- 进行字符串处理
- 筛选字符串
- 实现最小编辑距离算法，用于字符串比对并寻找最优编辑路径
- 理解动态规划的工作原理

同类系统应用十分广泛。
举个例子，如果你输入 **"I am lerningg"**，系统很大概率能判断出你实际想输入的是 **"learning"**，如图1所示。

<div style="width:image width px; font-size:100%; text-align:center;"><img src='auto-correct.png' alt="alternate text" width="width" height="height" style="width:300px;height:250px;" /> Figure 1 </div>

<a name='0-1'></a>
#### 0.1 编辑距离（Edit Distance）

在本作业中，你将实现能够纠正与目标词相差 1 个和 2 个编辑距离的单词的模型。
- 当我们需要对一个单词进行 n 次编辑才能将其变为另一个单词时，我们称这两个单词之间的编辑距离为 n。

一次编辑可以包含以下操作之一：

- 删除（移除一个字母）：'hat' => 'at, ha, ht'
- 交换（交换两个相邻字母）：'eta' => 'eat, tea, ...'
- 替换（将一个字母改为另一个字母）：'jat' => 'hat, rat, cat, mat, ...'
- 插入（添加一个字母）：'te' => 'the, ten, ate, ...'

你将使用上述四种方法来实现一个自动拼写校正器（Auto-correct）。
- 为此，你需要计算在给定输入条件下，某个特定单词是正确的概率。

你将要实现的这个自动拼写校正器最早由 [Peter Norvig](https://en.wikipedia.org/wiki/Peter_Norvig) 于 2007 年创建。
- 他的 [原始文章](https://norvig.com/spell-correct.html) 可能对本作业有参考价值。

我们拼写检查模型的目标是计算以下概率：

$$P(c|w) = \frac{P(w|c)\times P(c)}{P(w)} \tag{Eqn-1}$$

上述方程即为 [贝叶斯法则](https://en.wikipedia.org/wiki/Bayes%27_theorem)。
- 方程 1 表示：某个单词是正确的概率 $P(c|w)$，等于在它是正确的前提下出现特定单词 $w$ 的概率 $P(w|c)$，乘以该单词在一般情况下为正确的先验概率 $P(c)$，再除以该单词 $w$ 在一般情况下出现的概率 $P(w)$。
- 为了计算方程 1，你首先需要导入一个数据集，然后利用该数据集计算出所有所需的概率。

<a name='1'></a>
# Part 1: Data Preprocessing 

In [28]:
import re
from collections import Counter
import numpy as np
import pandas as pd

和其他所有机器学习任务一样，首要步骤就是处理数据集。
- 很多课程会直接提供预处理完毕的数据。
- 但在真实场景中搭建NLP系统时，需要自行加载并处理数据集。
- 接下来我们就实操体验真实场景下的数据预处理流程！

你的第一项任务：读取目录下名为 **'shakespeare.txt'** 的文件。
你可以通过菜单栏 `File ==> Open` 打开查看这份文件。

<a name='ex-1'></a>
### 练习1
实现函数 `process_data`，功能要求：
1）读取语料文本文件
2）将所有文本转为小写
3）返回单词组成的列表

#### 可选建议与提示
- 如果你希望获得更贴近真实工程场景的练习体验，暂时不要查看下方提示，尝试自行上网检索资料推导出解决方案。
- 需要少量指引时，可以鼠标点击绿色的「通用提示」区域展开阅读。
- 如果陷入瓶颈、运行结果达不到预期，可以点开绿色「详细提示」，获取完成该函数每一步操作对应的指引。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>General Hints</b></font>
</summary>
<p>
    
General Hints to get started
<ul>
    <li>Python <a href="https://docs.python.org/3/tutorial/inputoutput.html">input and output<a></li>
    <li>Python <a href="https://docs.python.org/3/library/re.html" >'re' documentation </a> </li>
</ul>
</p>


<details>    
<summary>
    <font size="3" color="darkgreen"><b>Detailed Hints</b></font>
</summary>
<p>     
Detailed hints if you're stuck
<ul>
    <li>Use 'with' syntax to read a file</li>
    <li>Decide whether to use 'read()' or 'readline().  What's the difference?</li>
    <li>Choose whether to use either str.lower() or str.lowercase().  What is the difference?</li>
    <li>Use re.findall(pattern, string)</li>
    <li>Look for the "Raw String Notation" section in the Python 're' documentation to understand the difference between r'\W', r'\W' and '\\W'. </li>
    <li>For the pattern, decide between using '\s', '\w', '\s+' or '\w+'.  What do you think are the differences?</li>
</ul>
</p>


In [29]:
# UNQ_C1 (唯一单元格标识，请勿修改)
# 计分函数：process_data
def process_data(file_name):
    """
    输入：
        file_name：当前目录下的文本文件名，读取该文件
    输出：
        words：列表，包含语料中全部转为小写后的单词
    """
    words = [] # 正确填充该变量并返回

    ### 代码开始 ###
    with open(file_name, 'r', encoding='utf-8') as f:
        text = f.read()
        # 使用正则表达式匹配单词，并将其转换为小写
        words = re.findall(r'\b\w+\b', text.lower())
    ### 代码结束 ###

    return words

Note, in the following cell, 'words' is converted to a python `set`. This eliminates any duplicate entries.

In [30]:
#DO NOT MODIFY THIS CELL
word_l = process_data('shakespeare.txt')
vocab = set(word_l)  # this will be your new vocabulary
print(f"The first ten words in the text are: \n{word_l[0:10]}")
print(f"There are {len(vocab)} unique words in the vocabulary.")

The first ten words in the text are: 
['o', 'for', 'a', 'muse', 'of', 'fire', 'that', 'would', 'ascend', 'the']
There are 6116 unique words in the vocabulary.


#### Expected Output
```Python
The first ten words in the text are: 
['o', 'for', 'a', 'muse', 'of', 'fire', 'that', 'would', 'ascend', 'the']
There are 6116 unique words in the vocabulary.
```

<a name='ex-2'></a>
### 练习2

实现一个 `get_count` 函数，该函数返回一个字典
- 字典的键是单词
- 每个单词对应的值是该单词在语料中出现的次数。

例如，给定如下句子：**"I am happy because I am learning"**，你的字典应当返回如下结果：
<table style="width:20%">

  <tr>
    <td> <b>键 </b>  </td>
    <td> <b>值 </b> </td> 


  </tr>
  <tr>
    <td> I  </td>
    <td> 2</td> 
 
  </tr>
   
  <tr>
    <td>am</td>
    <td>2</td> 
  </tr>

  <tr>
    <td>happy</td>
    <td>1</td> 
  </tr>
  
   <tr>
    <td>because</td>
    <td>1</td> 
  </tr>
  
   <tr>
    <td>learning</td>
    <td>1</td> 
  </tr>
</table>

**任务要求**：
实现 `get_count` 函数，返回一个字典，字典键为单词，值是单词在列表内出现的次数。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>Try implementing this using a for loop and a regular dictionary. This may be good practice for similar coding interview questions</li>
    <li>You can also use defaultdict instead of a regualr dictionary, along with the for loop</li>
    <li>Otherwise, to skip using a for loop, you can use Python's <a href="https://docs.python.org/3.7/library/collections.html#collections.Counter" > Counter class</a> </li>
</ul>
</p>

In [31]:
# UNQ_C2 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# UNIT TEST COMMENT: Candidate for Table Driven Tests
# GRADED FUNCTION: get_count
def get_count(word_l):
    '''
    Input:
        word_l: a set of words representing the corpus. 
    Output:
        word_count_dict: The wordcount dictionary where key is the word and value is its frequency.
    '''
    
    word_count_dict = {}  # fill this with word counts
    ### START CODE HERE 
    for word in word_l:
        word_count_dict[word] = word_count_dict.get(word, 0) + 1
    ### END CODE HERE ###
 
    return word_count_dict

In [32]:
#DO NOT MODIFY THIS CELL
word_count_dict = get_count(word_l)
print(f"There are {len(word_count_dict)} key values pairs")
print(f"The count for the word 'thee' is {word_count_dict.get('thee',0)}")

There are 6116 key values pairs
The count for the word 'thee' is 240



#### Expected Output
```Python
There are 6116 key values pairs
The count for the word 'thee' is 240
```

<a name='ex-3'></a>
### 练习3

给定单词计数字典，计算：从语料中随机抽取一个单词时，各个单词出现的概率。

$$P(w_i) = \frac{C(w_i)}{M} \tag{公式2}$$

其中：
$C(w_i)$ 代表单词 $w_i$ 在语料中出现的总次数；
$M$ 代表语料中所有单词的总数。

举个例子，在句子 **'I am happy because I am learning'** 中，单词 `am` 的概率为：

$$P(am) = \frac{C(w_i)}{M} = \frac {2}{7} \tag{公式3}$$

**任务要求：**
实现 `get_probs` 函数，用于求解单词在样本中出现的概率。函数返回字典：键为单词，值为该单词在语料中出现的概率。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
General advice
<ul>
    <li> Use dictionary.values() </li>
    <li> Use sum() </li>
    <li> The cardinality (number of words in the corpus should be equal to len(word_l).  You will calculate this same number, but using the word count dictionary.</li>
</ul>
    
If you're using a for loop:
<ul>
    <li> Use dictionary.keys() </li>
</ul>
    
If you're using a dictionary comprehension:
<ul>
    <li>Use dictionary.items() </li>
</ul>
</p>


In [33]:
# UNQ_C3 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: get_probs
def get_probs(word_count_dict):
    '''
    Input:
        word_count_dict: The wordcount dictionary where key is the word and value is its frequency.
    Output:
        probs: A dictionary where keys are the words and the values are the probability that a word will occur. 
    '''
    probs = {}  # return this variable correctly
    
    ### START CODE HERE ###
    M = sum(word_count_dict.values())  # Total number of words
    for word, count in word_count_dict.items():
        probs[word] = count / M  # Calculate probability
    ### END CODE HERE ###

    return probs

In [34]:
#DO NOT MODIFY THIS CELL
probs = get_probs(word_count_dict)
print(f"Length of probs is {len(probs)}")
print(f"P('thee') is {probs['thee']:.4f}")

Length of probs is 6116
P('thee') is 0.0045


#### Expected Output

```Python
Length of probs is 6116
P('thee') is 0.0045
```

<a name='2'></a>
# 第2部分：字符串操作

现在你已经计算出语料中所有单词的 $P(w_i)$。接下来你将编写若干字符串处理函数，用来对错误拼写的字符串进行编辑，生成正确拼写的候选单词。本节需要实现四个函数：

* `delete_letter`：输入一个单词，返回**删除任意一个字符后**得到的全部字符串。
* `switch_letter`：输入一个单词，返回**交换任意一对相邻字符位置**后得到的全部字符串。
* `replace_letter`：输入一个单词，返回**将任意一个字符替换为其他不同字母**后得到的全部字符串。
* `insert_letter`：输入一个单词，返回**在任意位置插入一个新字符**后得到的全部字符串。


#### 列表推导式

在 Python 中处理字符串与列表时常会用到一项语法特性：**列表推导式（list comprehensions）**。下方将要介绍的程序逻辑均采用列表推导式实现。当然，只要最终运行结果一致，你也可以选用其他实现方式。

后续内容会详细讲解如何使用列表推导式，并完成对应函数的编写。如果你已经熟练掌握 Python，可直接跳过相关提示，着手实现函数。

Python 的列表推导式能够把循环逻辑内嵌在列表定义当中，将多行代码压缩成一行。如果你尚不熟悉这种写法，相比普通 for 循环，第一眼看上去会觉得语序有些别扭。

<div style="width:image width px; font-size:100%; text-align:center;"><img src='GenericListComp3.PNG' alt="alternate text" width="width" height="height"  style="width:800px;height:400px;"/> Figure 2 </div>

The diagram above shows that the components of a list comprehension are the same components you would find in a typical for loop that appends to a list, but in a different order. With that in mind, we'll continue the specifics of this assignment. We will be very descriptive for the first function, `deletes()`, and less so in later functions as you become familiar with list comprehensions.

<a name='ex-4'></a>
### 练习4

**delete_letter() 实现要求：**
实现 `delete_letter()` 函数：输入一个单词，返回删除其中任意一个字符后得到的字符串列表。

举个例子，输入单词 **nice**，输出结果集合为：{'ice', 'nce', 'nic', 'nie'}。

**步骤1：** 创建 `splits` 列表。它包含将单词切分为左侧字符串 L 和右侧字符串 R 的所有切分方式。例如：
'nice' 的所有切分结果：`[('', 'nice'), ('n', 'ice'), ('ni', 'ce'), ('nic', 'e'), ('nice', '')]`

这种左右切分方式在接下来四个函数（删除、替换、交换、插入）中都会通用。

<div style="width:image width px; font-size:100%; text-align:center;"><img src='Splits1.PNG' alt="alternate text" width="width" height="height" style="width:650px;height:200px;" /> Figure 3 </div>

**步骤 2：** 这部分专门针对 `delete_letter`（删除字母）操作。在这里，我们要生成所有通过删除一个字符而得到的单词。  
这可以通过一行列表推导式来实现。你可以使用如下语法形式：  
`[f(a, b) for a, b in splits if condition]`  

以我们的示例 'nice' 为例，你会得到：  
['ice', 'nce', 'nie', 'nic']

<div style="width:image width px; font-size:100%; text-align:center;"><img src='ListComp2.PNG' alt="alternate text" width="width" height="height" style="width:550px;height:300px;" /> Figure 4 </div>

#### 辅助级别

请尝试按照以下辅助级别来完成此练习。  
- 我们希望这既能为你带来有意义的体验，又不会让你感到过于沮丧。
- 从第 1 级开始，然后根据需要进入第 2 级和第 3 级。

    - 第 1 级：尝试自己思考并独立实现。
    - 第 2 级：如果卡住了，可以点击“第 2 级提示”部分，获取一些入门提示。
    - 第 3 级：如果你希望获得更详细的指导，请点击“第 3 级提示”单元格，查看逐步操作说明。
    
- 如果仍然卡住，请参考上面“列表推导式”部分中的图片。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Level 2 Hints</b></font>
</summary>
<p>
<ul>
    <li><a href="" > Use array slicing like my_string[0:2] </a> </li>
    <li><a href="" > Use list comprehensions or for loops </a> </li>
</ul>
</p>


<details>    
<summary>
    <font size="3" color="darkgreen"><b>Level 3 Hints</b></font>
</summary>
<p>
<ul>
    <li>splits: Use array slicing, like my_str[0:2], to separate a string into two pieces.</li>
    <li>Do this in a loop or list comprehension, so that you have a list of tuples.
    <li> For example, "cake" can get split into "ca" and "ke". They're stored in a tuple ("ca","ke"), and the tuple is appended to a list.  We'll refer to these as L and R, so the tuple is (L,R)</li>
    <li>When choosing the range for your loop, if you input the word "cans" and generate the tuple  ('cans',''), make sure to include an if statement to check the length of that right-side string (R) in the tuple (L,R) </li>
    <li>deletes: Go through the list of tuples and combine the two strings together. You can use the + operator to combine two strings</li>
    <li>When combining the tuples, make sure that you leave out a middle character.</li>
    <li>Use array slicing to leave out the first character of the right substring.</li>
</ul>
</p>

In [35]:
# UNQ_C4 (唯一单元格标识符，请勿编辑)
# 单元测试注释：适用于表驱动测试的候选
# 计分函数：deletes
def delete_letter(word, verbose=False):
    '''
    输入参数：
        word: 你将为其生成词汇表中所有可能缺失一个字符的单词的字符串/单词
    输出：
        delete_l: 通过从 word 中删除 1 个字符而获得的所有可能字符串的列表
    '''
    
    delete_l = []
    split_l = []
    
    ### 在此处开始编写代码 ###
    split_l = [(word[:i],word[i:]) for i in range(0,len(word))]

    delete_l = [L + R[1:] for L, R in split_l if R]
    ### 在此处结束代码 ###


    if verbose: print(f"输入单词 {word}, \nsplit_l = {split_l}, \ndelete_l = {delete_l}")

    return delete_l

In [36]:
delete_word_l = delete_letter(word="cans",
                        verbose=True)

输入单词 cans, 
split_l = [('', 'cans'), ('c', 'ans'), ('ca', 'ns'), ('can', 's')], 
delete_l = ['ans', 'cns', 'cas', 'can']


#### 预期输出
```CPP
注意：你得到的 split_l 结果可能略有不同

输入单词 cans, 
split_l = [('', 'cans'), ('c', 'ans'), ('ca', 'ns'), ('can', 's')], 
delete_l = ['ans', 'cns', 'cas', 'can']

#### 备注 1
- 请注意，其中包含了额外的元组 `('cans', '')`。
- 只要你检查了元组 (L,R) 中右侧子字符串的长度，这不会有问题。
- 你能解释为什么这对于删除字符串列表（delete_l）会得到相同的结果吗？

```CPP
输入单词 cans, 
split_l = [('', 'cans'), ('c', 'ans'), ('ca', 'ns'), ('can', 's'), ('cans', '')], 
delete_l = ['ans', 'cns', 'cas', 'can']

#### 备注 2
如果你最终得到的单词与输入单词相同，就像这样：

```Python
输入单词 cans, 
split_l = [('', 'cans'), ('c', 'ans'), ('ca', 'ns'), ('can', 's'), ('cans', '')], 
delete_l = ['ans', 'cns', 'cas', 'can', 'cans']

In [37]:
# test # 2
print(f"Number of outputs of delete_letter('at') is {len(delete_letter('at'))}")

Number of outputs of delete_letter('at') is 2


#### Expected output

```CPP
Number of outputs of delete_letter('at') is 2
```

<a name='ex-5'></a>
### 练习 5

**switch_letter() 的说明**：现在实现一个函数，用于交换单词中的两个字母。它接受一个单词作为输入，并返回所有可能交换**相邻**两个字母后得到的单词列表。
- 例如，给定单词 'eta'，它返回 {'eat', 'tea'}，但不会返回 'ate'。

**步骤 1：** 与 delete_letter() 中的步骤 1 相同。  
**步骤 2：** 使用列表推导式或 for 循环，通过交换相邻字母来形成新字符串。其形式为：  
`[f(L,R) for L, R in splits if condition]`，其中 'condition' 将在给定的迭代中测试 R 的长度。详见下文。

<div style="width:image width px; font-size:100%; text-align:center;"><img src='Switches1.PNG' alt="alternate text" width="width" height="height" style="width:600px;height:200px;"/> Figure 5 </div>      

#### 难度级别

尝试按照以下难度级别来完成此练习。  
- 第 1 级：尝试自己思考并独立实现。  
- 第 2 级：如果卡住了，可以点击“第 2 级提示”部分，获取一些入门提示。  
- 第 3 级：如果你希望获得更详细的指导，请点击“第 3 级提示”单元格，查看逐步操作说明。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>第 2 级提示</b></font>
</summary>
<p>
<ul>
    <li><a href="" > 使用数组切片，例如 my_string[0:2] </a> </li>
    <li><a href="" > 使用列表推导式或 for 循环 </a> </li>
    <li>要进行交换，可以将整个单词视为分成 4 个不同的部分。在一张纸上写出 'cupcakes'，看看如何将其拆分为 ('cupc', 'k', 'a', 'es')</li>
</ul>
</p>
</details>

<details>    
<summary>
    <font size="3" color="darkgreen"><b>第 3 级提示</b></font>
</summary>
<p>
<ul>
    <li>拆分：使用数组切片，例如 my_str[0:2]，将字符串分成两部分。</li>
    <li>拆分操作与 delete_letter 中的拆分操作相同。</li>
    <li>要执行交换，请遍历元组列表并将四个字符串组合在一起。你可以使用 + 运算符来组合字符串。</li>
    <li>这四个字符串分别是：拆分元组中的左侧子字符串，接着是右侧子字符串的第一个字符（索引 1），然后是右侧子字符串的第零个字符（索引 0），最后是右侧子字符串的剩余部分。</li>
    <li>与 delete_letter 不同，你需要确保右侧子字符串至少达到最小长度。要了解原因，请回顾前面的提示要点（就在此条之前的那一条）。</li>
</ul>
</p>
</details>

In [38]:
# UNQ_C5 (唯一单元格标识符，请勿编辑)
# 单元测试注释：适用于表驱动测试的候选
# 计分函数：switches
def switch_letter(word, verbose=False):
    '''
    输入参数：
        word: 输入字符串
    输出：
        switches: 所有可能交换一个相邻字符后得到的字符串列表
    ''' 
    
    switch_l = []
    split_l = []

    def switch(a,b):
        a,b = b,a
        return a,b

    
    ### 在此处开始编写代码 ###
    split_l = [(word[:i],word[i:]) for i in range(0,len(word))]

    for L, R in split_l:
        # 只有右边至少2个字符，才可以交换相邻字符
        if len(R) >= 2:
            # 交换R的前两个字符，拼接新单词
            switched = L + R[1] + R[0] + R[2:]
            switch_l.append(switched)



    
    ### 在此处结束代码 ###

    
    if verbose: print(f"输入单词 = {word} \nsplit_l = {split_l} \nswitch_l = {switch_l}") 

    return switch_l

In [39]:
switch_word_l = switch_letter(word="eta",
                         verbose=True)

输入单词 = eta 
split_l = [('', 'eta'), ('e', 'ta'), ('et', 'a')] 
switch_l = ['tea', 'eat']


#### Expected output

```Python
Input word = eta 
split_l = [('', 'eta'), ('e', 'ta'), ('et', 'a')] 
switch_l = ['tea', 'eat']
```

#### Note 1

You may get this:
```Python
Input word = eta 
split_l = [('', 'eta'), ('e', 'ta'), ('et', 'a'), ('eta', '')] 
switch_l = ['tea', 'eat']
```
- Notice how it has the extra tuple `('eta', '')`.
- This is also correct.
- Can you think of why this is the case?

#### Note 2

If you get an error
```Python
IndexError: string index out of range
```
- Please see if you have checked the length of the strings when switching characters.

In [40]:
# test # 2
print(f"Number of outputs of switch_letter('at') is {len(switch_letter('at'))}")

Number of outputs of switch_letter('at') is 1


#### Expected output

```CPP
Number of outputs of switch_letter('at') is 1
```

<a name='ex-6'></a>
### 练习 6
**replace_letter() 的说明**：现在实现一个函数，该函数接受一个单词并返回一个列表，其中包含原单词中 **替换一个字母** 后得到的所有字符串。

**步骤 1：** 与 `delete_letter()` 中的步骤 1 相同。

**步骤 2：** 使用列表推导式或 for 循环，通过替换字母来形成新字符串。其形式可以为：  
`[f(a,b,c) for a, b in splits if condition for c in string]`  注意第二个 for 循环的使用。  
在本例程中，预期一个或多个替换结果会包含原单词。例如，将 'ear' 的第一个字母替换为 'e' 将返回 'ear'。

**步骤 3：** 从输出中移除原始输入单词。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>提示</b></font>
</summary>
<p>
<ul>
    <li>要从列表中移除一个单词，首先将其内容存储在一个 set() 中</li>
    <li>使用 set.discard('the_word') 来移除集合中的一个单词（如果该单词不存在于集合中，则不会抛出 KeyError。使用 set.remove('the_word') 在单词不存在于集合中时会抛出 KeyError）。</li>
</ul>
</p>
</details>

In [41]:
# UNQ_C6 (唯一单元格标识符，请勿编辑)
# 单元测试注释：适用于表驱动测试的候选
# 计分函数：replaces
def replace_letter(word, verbose=False):
    '''
    输入参数：
        word: 输入的字符串/单词 
    输出：
        replaces: 所有可能替换原单词中的一个字母后得到的字符串列表
    ''' 
    
    letters = 'abcdefghijklmnopqrstuvwxyz'
    replace_l = []
    split_l = []
    replace_set = set()
    
    ### 在此处开始编写代码 ###
    # 生成所有切分对
    split_l = [(word[:i], word[i:]) for i in range(len(word))]
    
    for L, R in split_l:
        if R:  # R非空，才有字符可以替换 R[0]
            original_char = R[0]
            for c in letters:
                # 不要替换成和原来一样的字符
                if c != original_char:
                    new_word = L + c + R[1:]
                    replace_set.add(new_word)
    ### 在此处结束代码 ###
    
    # 将集合转换回列表并排序，以便于查看
    replace_l = sorted(list(replace_set))
    
    if verbose: print(f"输入单词 = {word} \nsplit_l = {split_l} \nreplace_l {replace_l}")   
    
    return replace_l

In [42]:
replace_l = replace_letter(word='can',
                              verbose=True)

输入单词 = can 
split_l = [('', 'can'), ('c', 'an'), ('ca', 'n')] 
replace_l ['aan', 'ban', 'caa', 'cab', 'cac', 'cad', 'cae', 'caf', 'cag', 'cah', 'cai', 'caj', 'cak', 'cal', 'cam', 'cao', 'cap', 'caq', 'car', 'cas', 'cat', 'cau', 'cav', 'caw', 'cax', 'cay', 'caz', 'cbn', 'ccn', 'cdn', 'cen', 'cfn', 'cgn', 'chn', 'cin', 'cjn', 'ckn', 'cln', 'cmn', 'cnn', 'con', 'cpn', 'cqn', 'crn', 'csn', 'ctn', 'cun', 'cvn', 'cwn', 'cxn', 'cyn', 'czn', 'dan', 'ean', 'fan', 'gan', 'han', 'ian', 'jan', 'kan', 'lan', 'man', 'nan', 'oan', 'pan', 'qan', 'ran', 'san', 'tan', 'uan', 'van', 'wan', 'xan', 'yan', 'zan']


#### Expected Output**: 
```Python
Input word = can 
split_l = [('', 'can'), ('c', 'an'), ('ca', 'n')] 
replace_l ['aan', 'ban', 'caa', 'cab', 'cac', 'cad', 'cae', 'caf', 'cag', 'cah', 'cai', 'caj', 'cak', 'cal', 'cam', 'cao', 'cap', 'caq', 'car', 'cas', 'cat', 'cau', 'cav', 'caw', 'cax', 'cay', 'caz', 'cbn', 'ccn', 'cdn', 'cen', 'cfn', 'cgn', 'chn', 'cin', 'cjn', 'ckn', 'cln', 'cmn', 'cnn', 'con', 'cpn', 'cqn', 'crn', 'csn', 'ctn', 'cun', 'cvn', 'cwn', 'cxn', 'cyn', 'czn', 'dan', 'ean', 'fan', 'gan', 'han', 'ian', 'jan', 'kan', 'lan', 'man', 'nan', 'oan', 'pan', 'qan', 'ran', 'san', 'tan', 'uan', 'van', 'wan', 'xan', 'yan', 'zan']
```
- Note how the input word 'can' should not be one of the output words.

#### Note 1
If you get something like this:

```Python
Input word = can 
split_l = [('', 'can'), ('c', 'an'), ('ca', 'n'), ('can', '')] 
replace_l ['aan', 'ban', 'caa', 'cab', 'cac', 'cad', 'cae', 'caf', 'cag', 'cah', 'cai', 'caj', 'cak', 'cal', 'cam', 'cao', 'cap', 'caq', 'car', 'cas', 'cat', 'cau', 'cav', 'caw', 'cax', 'cay', 'caz', 'cbn', 'ccn', 'cdn', 'cen', 'cfn', 'cgn', 'chn', 'cin', 'cjn', 'ckn', 'cln', 'cmn', 'cnn', 'con', 'cpn', 'cqn', 'crn', 'csn', 'ctn', 'cun', 'cvn', 'cwn', 'cxn', 'cyn', 'czn', 'dan', 'ean', 'fan', 'gan', 'han', 'ian', 'jan', 'kan', 'lan', 'man', 'nan', 'oan', 'pan', 'qan', 'ran', 'san', 'tan', 'uan', 'van', 'wan', 'xan', 'yan', 'zan']
```
- Notice how split_l has an extra tuple `('can', '')`, but the output is still the same, so this is okay.

#### Note 2
If you get something like this:
```Python
Input word = can 
split_l = [('', 'can'), ('c', 'an'), ('ca', 'n'), ('can', '')] 
replace_l ['aan', 'ban', 'caa', 'cab', 'cac', 'cad', 'cae', 'caf', 'cag', 'cah', 'cai', 'caj', 'cak', 'cal', 'cam', 'cana', 'canb', 'canc', 'cand', 'cane', 'canf', 'cang', 'canh', 'cani', 'canj', 'cank', 'canl', 'canm', 'cann', 'cano', 'canp', 'canq', 'canr', 'cans', 'cant', 'canu', 'canv', 'canw', 'canx', 'cany', 'canz', 'cao', 'cap', 'caq', 'car', 'cas', 'cat', 'cau', 'cav', 'caw', 'cax', 'cay', 'caz', 'cbn', 'ccn', 'cdn', 'cen', 'cfn', 'cgn', 'chn', 'cin', 'cjn', 'ckn', 'cln', 'cmn', 'cnn', 'con', 'cpn', 'cqn', 'crn', 'csn', 'ctn', 'cun', 'cvn', 'cwn', 'cxn', 'cyn', 'czn', 'dan', 'ean', 'fan', 'gan', 'han', 'ian', 'jan', 'kan', 'lan', 'man', 'nan', 'oan', 'pan', 'qan', 'ran', 'san', 'tan', 'uan', 'van', 'wan', 'xan', 'yan', 'zan']
```
- Notice how there are strings that are 1 letter longer than the original word, such as `cana`.
- Please check for the case when there is an empty string `''`, and if so, do not use that empty string when setting replace_l.

In [43]:
# test # 2
print(f"Number of outputs of switch_letter('at') is {len(switch_letter('at'))}")

Number of outputs of switch_letter('at') is 1


#### Expected output
```CPP
Number of outputs of switch_letter('at') is 1
```

<a name='ex-7'></a>
### 练习 7

**insert_letter() 的说明**：现在实现一个函数，该函数接受一个单词并返回一个列表，其中包含在每个偏移位置插入一个字母后得到的所有字符串。

**步骤 1：** 与 `delete_letter()` 中的步骤 1 相同。

**步骤 2：** 这可以是一个列表推导式，其形式为：  
`[f(a,b,c) for a, b in splits if condition for c in string]`

In [44]:
# UNQ_C7 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# UNIT TEST COMMENT: Candidate for Table Driven Tests
# GRADED FUNCTION: inserts
def insert_letter(word, verbose=False):
    '''
    Input:
        word: the input string/word 
    Output:
        inserts: a set of all possible strings with one new letter inserted at every offset
    ''' 
    letters = 'abcdefghijklmnopqrstuvwxyz'
    insert_l = []
    split_l = []
    
    ### START CODE HERE ###
    split_l = [(word[:i], word[i:]) for i in range(len(word)+1)]

    for L, R in split_l:
        for c in letters:
            newword = L + c + R
            insert_l.append(newword)

         
                
    ### END CODE HERE ###


    if verbose: print(f"Input word {word} \nsplit_l = {split_l} \ninsert_l = {insert_l}")
    
    return insert_l

In [45]:
insert_l = insert_letter('at', True)
print(f"Number of strings output by insert_letter('at') is {len(insert_l)}")

Input word at 
split_l = [('', 'at'), ('a', 't'), ('at', '')] 
insert_l = ['aat', 'bat', 'cat', 'dat', 'eat', 'fat', 'gat', 'hat', 'iat', 'jat', 'kat', 'lat', 'mat', 'nat', 'oat', 'pat', 'qat', 'rat', 'sat', 'tat', 'uat', 'vat', 'wat', 'xat', 'yat', 'zat', 'aat', 'abt', 'act', 'adt', 'aet', 'aft', 'agt', 'aht', 'ait', 'ajt', 'akt', 'alt', 'amt', 'ant', 'aot', 'apt', 'aqt', 'art', 'ast', 'att', 'aut', 'avt', 'awt', 'axt', 'ayt', 'azt', 'ata', 'atb', 'atc', 'atd', 'ate', 'atf', 'atg', 'ath', 'ati', 'atj', 'atk', 'atl', 'atm', 'atn', 'ato', 'atp', 'atq', 'atr', 'ats', 'att', 'atu', 'atv', 'atw', 'atx', 'aty', 'atz']
Number of strings output by insert_letter('at') is 78


#### Expected output

```Python
Input word at 
split_l = [('', 'at'), ('a', 't'), ('at', '')] 
insert_l = ['aat', 'bat', 'cat', 'dat', 'eat', 'fat', 'gat', 'hat', 'iat', 'jat', 'kat', 'lat', 'mat', 'nat', 'oat', 'pat', 'qat', 'rat', 'sat', 'tat', 'uat', 'vat', 'wat', 'xat', 'yat', 'zat', 'aat', 'abt', 'act', 'adt', 'aet', 'aft', 'agt', 'aht', 'ait', 'ajt', 'akt', 'alt', 'amt', 'ant', 'aot', 'apt', 'aqt', 'art', 'ast', 'att', 'aut', 'avt', 'awt', 'axt', 'ayt', 'azt', 'ata', 'atb', 'atc', 'atd', 'ate', 'atf', 'atg', 'ath', 'ati', 'atj', 'atk', 'atl', 'atm', 'atn', 'ato', 'atp', 'atq', 'atr', 'ats', 'att', 'atu', 'atv', 'atw', 'atx', 'aty', 'atz']
Number of strings output by insert_letter('at') is 78
```

#### Note 1

If you get a split_l like this:
```Python
Input word at 
split_l = [('', 'at'), ('a', 't')] 
insert_l = ['aat', 'bat', 'cat', 'dat', 'eat', 'fat', 'gat', 'hat', 'iat', 'jat', 'kat', 'lat', 'mat', 'nat', 'oat', 'pat', 'qat', 'rat', 'sat', 'tat', 'uat', 'vat', 'wat', 'xat', 'yat', 'zat', 'aat', 'abt', 'act', 'adt', 'aet', 'aft', 'agt', 'aht', 'ait', 'ajt', 'akt', 'alt', 'amt', 'ant', 'aot', 'apt', 'aqt', 'art', 'ast', 'att', 'aut', 'avt', 'awt', 'axt', 'ayt', 'azt']
Number of strings output by insert_letter('at') is 52
```
- Notice that split_l is missing the extra tuple ('at', '').  For insertion, we actually **WANT** this tuple.
- The function is not creating all the desired output strings.
- Check the range that you use for the for loop.

#### Note 2
If you see this:
```Python
Input word at 
split_l = [('', 'at'), ('a', 't'), ('at', '')] 
insert_l = ['aat', 'bat', 'cat', 'dat', 'eat', 'fat', 'gat', 'hat', 'iat', 'jat', 'kat', 'lat', 'mat', 'nat', 'oat', 'pat', 'qat', 'rat', 'sat', 'tat', 'uat', 'vat', 'wat', 'xat', 'yat', 'zat', 'aat', 'abt', 'act', 'adt', 'aet', 'aft', 'agt', 'aht', 'ait', 'ajt', 'akt', 'alt', 'amt', 'ant', 'aot', 'apt', 'aqt', 'art', 'ast', 'att', 'aut', 'avt', 'awt', 'axt', 'ayt', 'azt']
Number of strings output by insert_letter('at') is 52
```

- Even though you may have fixed the split_l so that it contains the tuple `('at', '')`, notice that you're still missing some output strings.
    - Notice that it's missing strings such as 'ata', 'atb', 'atc' all the way to 'atz'.
- To fix this, make sure that when you set insert_l, you allow the use of the empty string `''`.

In [46]:
# test # 2
print(f"Number of outputs of insert_letter('at') is {len(insert_letter('at'))}")

Number of outputs of insert_letter('at') is 78


#### Expected output

```CPP
Number of outputs of insert_letter('at') is 78
```

<a name='3'></a>

# 第 3 部分：组合编辑操作

现在你已经实现了字符串操作函数，接下来将创建两个函数，它们能够针对给定的字符串，返回该字符串所有可能的单次编辑和双次编辑结果。这两个函数分别是 `edit_one_letter()` 和 `edit_two_letters()`。

<a name='3-1'></a>
## 3.1 编辑一个字母

<a name='ex-8'></a>
### 练习 8

**说明**：实现 `edit_one_letter` 函数，以获取与某个单词相差一次编辑的所有可能结果。这些编辑操作包括替换、插入、删除，以及可选的交换操作。你应使用之前已实现的函数来完成此函数。'switch' 函数是一种不太常见的编辑操作，因此其使用将由 `allow_switches` 输入参数来控制。

请注意，那些函数返回的是 *列表*，而本函数应返回一个 *Python 集合*。使用集合可以消除任何重复条目。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li> Each of the functions returns a list.  You can combine lists using the `+` operator. </li>
    <li> To get unique strings (avoid duplicates), you can use the set() function. </li>
</ul>
</p>


In [60]:
# UNQ_C8 (唯一单元格标识符，请勿编辑)
# 单元测试注释：适用于表驱动测试的候选
# 计分函数：edit_one_letter
def edit_one_letter(word, allow_switches = True):
    """
    输入参数：
        word: 我们将为其生成所有相差一次编辑的单词的字符串/单词
    输出：
        edit_one_set: 具有一次可能编辑的单词集合。请返回一个集合，而不是列表。
    """
    
    edit_one_set = set()
    
    ### 在此处开始编写代码 ###
    d_l= delete_letter(word=word)
    i_l= insert_letter(word=word)
    r_l = replace_letter(word=word)
    if allow_switches:
        s_l = switch_letter(word=word)

    all_l = d_l+i_l+ r_l + s_l

    for each in all_l:
        edit_one_set.add(each)
    ### 在此处结束代码 ###


    return edit_one_set

In [61]:
tmp_word = "at"
tmp_edit_one_set = edit_one_letter(tmp_word)
# turn this into a list to sort it, in order to view it
tmp_edit_one_l = sorted(list(tmp_edit_one_set))

print(f"input word {tmp_word} \nedit_one_l \n{tmp_edit_one_l}\n")
print(f"The type of the returned object should be a set {type(tmp_edit_one_set)}")
print(f"Number of outputs from edit_one_letter('at') is {len(edit_one_letter('at'))}")

input word at 
edit_one_l 
['a', 'aa', 'aat', 'ab', 'abt', 'ac', 'act', 'ad', 'adt', 'ae', 'aet', 'af', 'aft', 'ag', 'agt', 'ah', 'aht', 'ai', 'ait', 'aj', 'ajt', 'ak', 'akt', 'al', 'alt', 'am', 'amt', 'an', 'ant', 'ao', 'aot', 'ap', 'apt', 'aq', 'aqt', 'ar', 'art', 'as', 'ast', 'ata', 'atb', 'atc', 'atd', 'ate', 'atf', 'atg', 'ath', 'ati', 'atj', 'atk', 'atl', 'atm', 'atn', 'ato', 'atp', 'atq', 'atr', 'ats', 'att', 'atu', 'atv', 'atw', 'atx', 'aty', 'atz', 'au', 'aut', 'av', 'avt', 'aw', 'awt', 'ax', 'axt', 'ay', 'ayt', 'az', 'azt', 'bat', 'bt', 'cat', 'ct', 'dat', 'dt', 'eat', 'et', 'fat', 'ft', 'gat', 'gt', 'hat', 'ht', 'iat', 'it', 'jat', 'jt', 'kat', 'kt', 'lat', 'lt', 'mat', 'mt', 'nat', 'nt', 'oat', 'ot', 'pat', 'pt', 'qat', 'qt', 'rat', 'rt', 'sat', 'st', 't', 'ta', 'tat', 'tt', 'uat', 'ut', 'vat', 'vt', 'wat', 'wt', 'xat', 'xt', 'yat', 'yt', 'zat', 'zt']

The type of the returned object should be a set <class 'set'>
Number of outputs from edit_one_letter('at') is 129


#### Expected Output
```CPP
input word at 
edit_one_l 
['a', 'aa', 'aat', 'ab', 'abt', 'ac', 'act', 'ad', 'adt', 'ae', 'aet', 'af', 'aft', 'ag', 'agt', 'ah', 'aht', 'ai', 'ait', 'aj', 'ajt', 'ak', 'akt', 'al', 'alt', 'am', 'amt', 'an', 'ant', 'ao', 'aot', 'ap', 'apt', 'aq', 'aqt', 'ar', 'art', 'as', 'ast', 'ata', 'atb', 'atc', 'atd', 'ate', 'atf', 'atg', 'ath', 'ati', 'atj', 'atk', 'atl', 'atm', 'atn', 'ato', 'atp', 'atq', 'atr', 'ats', 'att', 'atu', 'atv', 'atw', 'atx', 'aty', 'atz', 'au', 'aut', 'av', 'avt', 'aw', 'awt', 'ax', 'axt', 'ay', 'ayt', 'az', 'azt', 'bat', 'bt', 'cat', 'ct', 'dat', 'dt', 'eat', 'et', 'fat', 'ft', 'gat', 'gt', 'hat', 'ht', 'iat', 'it', 'jat', 'jt', 'kat', 'kt', 'lat', 'lt', 'mat', 'mt', 'nat', 'nt', 'oat', 'ot', 'pat', 'pt', 'qat', 'qt', 'rat', 'rt', 'sat', 'st', 't', 'ta', 'tat', 'tt', 'uat', 'ut', 'vat', 'vt', 'wat', 'wt', 'xat', 'xt', 'yat', 'yt', 'zat', 'zt']

The type of the returned object should be a set <class 'set'>
Number of outputs from edit_one_letter('at') is 129
```

<a name='3-2'></a>
## 第 3.2 部分 编辑两个字母

<a name='ex-9'></a>
### 练习 9

现在你可以将其推广，实现对某个单词进行两次编辑。要做到这一点，你需要先获取某个单词所有可能的单次编辑结果，然后对每个修改后的单词再修改一次。

**说明**：实现 `edit_two_letters` 函数，该函数返回一个由相差两次编辑的单词组成的集合。请注意，基于 `edit_one_letter` 函数创建额外编辑时，可能会将某些单次编辑结果“恢复”为零次或一次编辑。这种情况在此是允许的，并在 `get_corrections` 中有所考虑。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>You will likely want to take the union of two sets.</li>
    <li>You can either use set.union() or use the '|' (or operator) to union two sets</li>
    <li>See the documentation <a href="https://docs.python.org/2/library/sets.html" > Python sets </a> for examples of using operators or functions of the Python set.</li>
</ul>
</p>


In [62]:
# UNQ_C9 (唯一单元格标识符，请勿编辑)
# 单元测试注释：适用于表驱动测试的候选
# 计分函数：edit_two_letters
def edit_two_letters(word, allow_switches = True):
    '''
    输入参数：
        word: 输入的字符串/单词
    输出：
        edit_two_set: 包含所有可能的两次编辑结果的字符串集合
    '''
    
    edit_two_set = set()
    
    ### 在此处开始编写代码 ###
    edit_one_set = edit_one_letter(word, allow_switches)
    for each in edit_one_set:
        edit_two_set.update(edit_one_letter(each, allow_switches))
    ### 在此处结束代码 ###

    
    return edit_two_set

In [63]:
tmp_edit_two_set = edit_two_letters("a")
tmp_edit_two_l = sorted(list(tmp_edit_two_set))
print(f"Number of strings with edit distance of two: {len(tmp_edit_two_l)}")
print(f"First 10 strings {tmp_edit_two_l[:10]}")
print(f"Last 10 strings {tmp_edit_two_l[-10:]}")
print(f"The data type of the returned object should be a set {type(tmp_edit_two_set)}")
print(f"Number of strings that are 2 edit distances from 'at' is {len(edit_two_letters('at'))}")

Number of strings with edit distance of two: 2654
First 10 strings ['', 'a', 'aa', 'aaa', 'aab', 'aac', 'aad', 'aae', 'aaf', 'aag']
Last 10 strings ['zv', 'zva', 'zw', 'zwa', 'zx', 'zxa', 'zy', 'zya', 'zz', 'zza']
The data type of the returned object should be a set <class 'set'>
Number of strings that are 2 edit distances from 'at' is 7154


#### Expected Output

```CPP
Number of strings with edit distance of two: 2654
First 10 strings ['', 'a', 'aa', 'aaa', 'aab', 'aac', 'aad', 'aae', 'aaf', 'aag']
Last 10 strings ['zv', 'zva', 'zw', 'zwa', 'zx', 'zxa', 'zy', 'zya', 'zz', 'zza']
The data type of the returned object should be a set <class 'set'>
Number of strings that are 2 edit distances from 'at' is 7154
```

<a name='3-3'></a>
## 第 3-3 部分：提供拼写建议

现在你将使用 `edit_two_letters` 函数来获取单词所有可能的两次编辑结果集合。然后，你将使用这些字符串来获取你最可能想要输入的单词，即你的打字建议。

<a name='ex-10'></a>
### 练习 10
**说明**：实现 `get_corrections`，该函数返回一个包含零到 n 个建议元组的列表，元组形式为 (word, probability_of_word)。

**步骤 1：** 为提供的单词生成建议：你将使用已开发的编辑函数。“建议算法”应遵循以下逻辑：
* 如果该单词在词汇表中，则建议该单词。
* 否则，如果 `edit_one_letter` 中有建议在词汇表中，则使用这些建议。
* 否则，如果 `edit_two_letters` 中有建议在词汇表中，则使用这些建议。
* 否则，建议输入单词本身。
* 其思路是，编辑次数越少的单词比编辑次数越多的单词更可能为正确单词。

注意：
- 一次或两次字母编辑可能会将字符串“恢复”为零次或一次编辑。该算法通过优先选择编辑距离更低的建议来考虑这一点。

#### 短路求值
在 Python 中，诸如 `and` 和 `or` 这样的逻辑运算有两个有用的特性。它们可以操作列表，并且具有 ['短路' 行为](https://docs.python.org/3/library/stdtypes.html)。请尝试以下操作：

In [64]:
# example of logical operation on lists or sets
print( [] and ["a","b"] )
print( [] or ["a","b"] )
#example of Short circuit behavior
val1 =  ["Most","Likely"] or ["Less","so"] or ["least","of","all"]  # selects first, does not evalute remainder
print(val1)
val2 =  [] or [] or ["least","of","all"] # continues evaluation until there is a non-empty list
print(val2)

[]
['a', 'b']
['Most', 'Likely']
['least', 'of', 'all']


逻辑运算符 `or` 可以非常简洁地用于实现建议算法。另外，也可以使用 if/then 结构。

**步骤 2**：创建一个 `best_words` 字典，其中“键”是一个建议词，“值”是该词在词汇表中的概率。如果该词不在词汇表中，则为其分配概率 0。

**步骤 3**：选择前 n 个最佳建议。实际建议数可能少于 n。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>提示</b></font>
</summary>
<p>
<ul>
    <li>edit_one_letter 和 edit_two_letters 返回 *Python 集合*。</li>
    <li>集合有一个方便的 <a href="https://docs.python.org/2/library/sets.html" > set.intersection </a> 功能。</li>
    <li>要找出字典中具有最高值的键，可以使用 Counter 字典从普通字典创建 Counter 对象，然后使用 Counter.most_common(n) 获取最常见的 n 个键。</li>
    <li>要找出两个集合的交集，可以使用 set.intersection 或 & 运算符。</li>
    <li>如果你对短路语法（如上所示）不太熟悉，可以随意使用 if else 语句代替。</li>
    <li>要使用 if 语句检查集合是否为空，可以使用 'if not x:' 语法。</li>
</ul>
</p>
</details>

In [ ]:
# UNQ_C10 (唯一单元格标识符，请勿编辑)
# 单元测试注释：适用于表驱动测试的候选
# 计分函数：get_corrections
def get_corrections(word, probs, vocab, n=2, verbose = False):
    '''
    输入参数：
        word: 用户输入的字符串，用于检查建议
        probs: 一个字典，将每个单词映射到其在语料库中的概率
        vocab: 包含所有词汇的集合
        n: 你希望返回的候选单词建议数量
    输出：
        n_best: 一个包含最可能的 n 个校正单词及其概率的元组列表
    '''
    
    suggestions = []
    n_best = []
    
    ### 在此处开始编写代码 ###
    
    ### 在此处结束代码 ###

    
    if verbose: print("输入的单词 = ", word, "\nsuggestions = ", suggestions)

    return n_best

In [53]:
# Test your implementation - feel free to try other words in my word
my_word = 'dys' 
tmp_corrections = get_corrections(my_word, probs, vocab, 2, verbose=True) # keep verbose=True
for i, word_prob in enumerate(tmp_corrections):
    print(f"word {i}: {word_prob[0]}, probability {word_prob[1]:.6f}")

print(f"data type of corrections {type(tmp_corrections)}")

entered word =  dys 
suggestions =  []
data type of corrections <class 'list'>


#### Expected Output
- Note: This expected output is for `my_word = 'dys'`. Also, keep `verbose=True`
```CPP
entered word =  dys 
suggestions =  {'days', 'dye'}
word 0: days, probability 0.000410
word 1: dye, probability 0.000019
data type of corrections <class 'list'>
```

<a name='4'></a>
# Part 4: Minimum Edit distance

Now that you have implemented your auto-correct, how do you evaluate the similarity between two strings? For example: 'waht' and 'what'

Also how do you efficiently find the shortest path to go from the word, 'waht' to the word 'what'?

You will implement a dynamic programming system that will tell you the minimum number of edits required to convert a string into another string.

<a name='4-1'></a>
### Part 4.1 Dynamic Programming

Dynamic Programming breaks a problem down into subproblems which can be combined to form the final solution. Here, given a string source[0..i] and a string target[0..j], we will compute all the combinations of substrings[i, j] and calculate their edit distance. To do this efficiently, we will use a table to maintain the previously computed substrings and use those to calculate larger substrings.

You have to create a matrix and update each element in the matrix as follows:  

$$\text{Initialization}$$

\begin{align}
D[0,0] &= 0 \\
D[i,0] &= D[i-1,0] + del\_cost(source[i]) \tag{4}\\
D[0,j] &= D[0,j-1] + ins\_cost(target[j]) \\
\end{align}


$$\text{Per Cell Operations}$$
\begin{align}
 \\
D[i,j] =min
\begin{cases}
D[i-1,j] + del\_cost\\
D[i,j-1] + ins\_cost\\
D[i-1,j-1] + \left\{\begin{matrix}
rep\_cost; & if src[i]\neq tar[j]\\
0 ; & if src[i]=tar[j]
\end{matrix}\right.
\end{cases}
\tag{5}
\end{align}

So converting the source word **play** to the target word **stay**, using an input cost of one, a delete cost of 1, and replace cost of 2 would give you the following table:
<table style="width:20%">

  <tr>
    <td> <b> </b>  </td>
    <td> <b># </b>  </td>
    <td> <b>s </b>  </td>
    <td> <b>t </b> </td> 
    <td> <b>a </b> </td> 
    <td> <b>y </b> </td> 
  </tr>
   <tr>
    <td> <b>  #  </b></td>
    <td> 0</td> 
    <td> 1</td> 
    <td> 2</td> 
    <td> 3</td> 
    <td> 4</td> 
 
  </tr>
  <tr>
    <td> <b>  p  </b></td>
    <td> 1</td> 
 <td> 2</td> 
    <td> 3</td> 
    <td> 4</td> 
   <td> 5</td>
  </tr>
   
  <tr>
    <td> <b> l </b></td>
    <td>2</td> 
    <td>3</td> 
    <td>4</td> 
    <td>5</td> 
    <td>6</td>
  </tr>

  <tr>
    <td> <b> a </b></td>
    <td>3</td> 
     <td>4</td> 
     <td>5</td> 
     <td>4</td>
     <td>5</td> 
  </tr>
  
   <tr>
    <td> <b> y </b></td>
    <td>4</td> 
      <td>5</td> 
     <td>6</td> 
     <td>5</td>
     <td>4</td> 
  </tr>
  

</table>



The operations used in this algorithm are 'insert', 'delete', and 'replace'. These correspond to the functions that you defined earlier: insert_letter(), delete_letter() and replace_letter(). switch_letter() is not used here.

The diagram below describes how to initialize the table. Each entry in D[i,j] represents the minimum cost of converting string source[0:i] to string target[0:j]. The first column is initialized to represent the cumulative cost of deleting the source characters to convert string "EER" to "". The first row is initialized to represent the cumulative cost of inserting the target characters to convert from "" to "NEAR".

<div style="width:image width px; font-size:100%; text-align:center;"><img src='EditDistInit4.PNG' alt="alternate text" width="width" height="height" style="width:1000px;height:400px;"/> Figure 6 Initializing Distance Matrix</div>     

Filling in the remainder of the table utilizes the 'Per Cell Operations' in the equation (5) above. Note, the diagram below includes in the table some of the 3 sub-calculations shown in light grey. Only 'min' of those operations is stored in the table in the `min_edit_distance()` function.

<div style="width:image width px; font-size:100%; text-align:center;"><img src='EditDistFill2.PNG' alt="alternate text" width="width" height="height" style="width:800px;height:400px;"/> Figure 7 Filling Distance Matrix</div>     

Note that the formula for $D[i,j]$ shown in the image is equivalent to:

\begin{align}
 \\
D[i,j] =min
\begin{cases}
D[i-1,j] + del\_cost\\
D[i,j-1] + ins\_cost\\
D[i-1,j-1] + \left\{\begin{matrix}
rep\_cost; & if src[i]\neq tar[j]\\
0 ; & if src[i]=tar[j]
\end{matrix}\right.
\end{cases}
\tag{5}
\end{align}

The variable `sub_cost` (for substitution cost) is the same as `rep_cost`; replacement cost.  We will stick with the term "replace" whenever possible.

Below are some examples of cells where replacement is used. This also shows the minimum path from the lower right final position where "EER" has been replaced by "NEAR" back to the start. This provides a starting point for the optional 'backtrace' algorithm below.

<div style="width:image width px; font-size:100%; text-align:center;"><img src='EditDistExample1.PNG' alt="alternate text" width="width" height="height" style="width:1200px;height:400px;"/> Figure 8 Examples Distance Matrix</div>    

<a name='ex-11'></a>
### Exercise 11

Again, the word "substitution" appears in the figure, but think of this as "replacement".

**Instructions**: Implement the function below to get the minimum amount of edits required given a source string and a target string. 

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>The range(start, stop, step) function excludes 'stop' from its output</li>
    <li><a href="" > words </a> </li>
</ul>
</p>


In [54]:
# UNQ_C11 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: min_edit_distance
def min_edit_distance(source, target, ins_cost = 1, del_cost = 1, rep_cost = 2):
    '''
    Input: 
        source: a string corresponding to the string you are starting with
        target: a string corresponding to the string you want to end with
        ins_cost: an integer setting the insert cost
        del_cost: an integer setting the delete cost
        rep_cost: an integer setting the replace cost
    Output:
        D: a matrix of len(source)+1 by len(target)+1 containing minimum edit distances
        med: the minimum edit distance (med) required to convert the source string to the target
    '''
    # use deletion and insert cost as  1
    m = len(source) 
    n = len(target) 
    #initialize cost matrix with zeros and dimensions (m+1,n+1) 
    D = np.zeros((m+1, n+1), dtype=int) 
    
    ### START CODE HERE (Replace instances of 'None' with your code) ###
    ### END CODE HERE ###

    return D, med

In [55]:
#DO NOT MODIFY THIS CELL
# testing your implementation 
source =  'play'
target = 'stay'
matrix, min_edits = min_edit_distance(source, target)
print("minimum edits: ",min_edits, "\n")
idx = list('#' + source)
cols = list('#' + target)
df = pd.DataFrame(matrix, index=idx, columns= cols)
print(df)

NameError: name 'med' is not defined

**Expected Results:**  

```CPP
minimum edits:  4
    
   #  s  t  a  y
#  0  1  2  3  4
p  1  2  3  4  5
l  2  3  4  5  6
a  3  4  5  4  5
y  4  5  6  5  4
```

In [ ]:
#DO NOT MODIFY THIS CELL
# testing your implementation 
source =  'eer'
target = 'near'
matrix, min_edits = min_edit_distance(source, target)
print("minimum edits: ",min_edits, "\n")
idx = list(source)
idx.insert(0, '#')
cols = list(target)
cols.insert(0, '#')
df = pd.DataFrame(matrix, index=idx, columns= cols)
print(df)

**Expected Results**  
```CPP
minimum edits:  3 

   #  n  e  a  r
#  0  1  2  3  4
e  1  2  1  2  3
e  2  3  2  3  4
r  3  4  3  4  3
```

We can now test several of our routines at once:

In [ ]:
source = "eer"
targets = edit_one_letter(source,allow_switches = False)  #disable switches since min_edit_distance does not include them
for t in targets:
    _, min_edits = min_edit_distance(source, t,1,1,1)  # set ins, del, sub costs all to one
    if min_edits != 1: print(source, t, min_edits)

**Expected Results**  
```CPP
(empty)
```

The 'replace()' routine utilizes all letters a-z one of which returns the original word.

In [ ]:
source = "eer"
targets = edit_two_letters(source,allow_switches = False) #disable switches since min_edit_distance does not include them
for t in targets:
    _, min_edits = min_edit_distance(source, t,1,1,1)  # set ins, del, sub costs all to one
    if min_edits != 2 and min_edits != 1: print(source, t, min_edits)

**Expected Results**  
```CPP
eer eer 0
```

We have to allow single edits here because some two_edits will restore a single edit.

# Submission
Make sure you submit your assignment before you modify anything below


<a name='5'></a>

# Part 5: Optional - Backtrace


Once you have computed your matrix using minimum edit distance, how would find the shortest path from the top left corner to the bottom right corner? 

Note that you could use backtrace algorithm.  Try to find the shortest path given the matrix that your `min_edit_distance` function returned.

You can use these [lecture slides on minimum edit distance](https://web.stanford.edu/class/cs124/lec/med.pdf) by Dan Jurafsky to learn about the algorithm for backtrace.

In [ ]:
# Experiment with back trace - insert your code here



#### References
- Dan Jurafsky - Speech and Language Processing - Textbook
- This auto-correct explanation was first done by Peter Norvig in 2007 